# How to use `gtfs_utils_v2`

In [4]:
import os
os.environ["CALITP_BQ_MAX_BYTES"] = str(400_000_000_000)

import datetime
import geopandas as gpd
import pandas as pd

from siuba import *

from shared_utils import gtfs_utils_v2
analysis_date = datetime.date(2023, 1, 17)

In [32]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

## Step 1: Airtable's `dim_gtfs_datasets` with `fct_daily_scheduled_feeds`

* Filter out ones that are not deprecated (use `data_quality_pipeline`)
* Allow any other custom filtering to be done, such as the default of getting scheduled data only
* Analysts now should look at output and decide if there's additional filtering needed. 
* Once filtering is done, input the df in to merge with `fct_daily_feeds` to get `feed_keys`
* Use `feed_key` to traverse all the other tables

## `feed_options`
* "customer_facing"
* "use_subfeeds", 
* "current_feeds"
* "include_precursor"
* "include_precursor_and_future"

In [33]:
df1 = gtfs_utils_v2.get_transit_organizations_gtfs_dataset_keys(
    keep_cols = None,
    custom_filtering = None,
    get_df = True)

In [34]:
df1.shape

(4119, 21)

In [36]:
df1.name.nunique()

566

In [37]:
df1.gtfs_dataset_key.nunique()

4119

In [38]:
df1.sample(3)

,gtfs_dataset_key,source_record_id,name,type,regional_feed_type,backdated_regional_feed_type,uri,future_uri,deprecated_date,data_quality_pipeline,manual_check__link_to_dataset_on_website,manual_check__accurate_shapes,manual_check__data_license,manual_check__authentication_acceptable,manual_check__stable_url,manual_check__localized_stop_tts,manual_check__grading_scheme_v1,base64_url,_is_current,_valid_from,_valid_to
3299,bc1f8670a7e5d0f93763ae85b8ac0afd,recj2HiEr7mbHFshA,SolTrans TripUpdates,trip_updates,None,Regional Precursor Feed,https://soltrans.connexionz.net/rtt/public/utility/gtfsrealtime.aspx/tripupdate2,None,None,True,Unknown,Unknown,Unknown,Yes,Yes,Unknown,Unknown,aHR0cHM6Ly9zb2x0cmFucy5jb25uZXhpb256Lm5ldC9ydHQvcHVibGljL3V0aWxpdHkvZ3Rmc3JlYWx0aW1lLmFzcHgvdHJpcHVwZGF0ZTI=,False,2022-10-18 00:00:00+00:00,2022-11-21 23:59:59.999999+00:00
570,5c73501029a4ea40ca653cfd8705c261,recAIrlYDHL3DW533,Bay Area 511 SolTrans Schedule,schedule,None,Regional Subfeed,http://api.511.org/transit/datafeeds?api_key={{ MTC_511_API_KEY}}&operator_id=ST,None,None,True,None,None,None,None,None,None,None,aHR0cDovL2FwaS41MTEub3JnL3RyYW5zaXQvZGF0YWZlZWRzP29wZXJhdG9yX2lkPVNU,False,2022-08-30 00:00:00+00:00,2022-08-30 23:59:59.999999+00:00
637,fec9627db106ee143b263f5a5557bb84,recxvieMsP0pWdxw7,Bay Area 511 Golden Gate Trip Updates,trip_updates,Regional Subfeed,Regional Subfeed,https://api.511.org/transit/tripupdates?api_key={{ MTC_511_API_KEY}}&agency=GG,None,None,True,None,None,Other-Permissive,Yes,Yes - Regional Partner Website,None,None,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cGRhdGVzP2FnZW5jeT1HRw==,False,2022-12-10 00:00:00+00:00,2023-01-13 23:59:59.999999+00:00


In [5]:
full_df = gtfs_utils_v2.schedule_daily_feed_to_organization(
    selected_date = analysis_date,
    keep_cols = None,
    get_df = True,
    feed_option = "")

In [6]:
full_df.shape

(227, 9)

In [28]:
full_df.is_future.value_counts()

False    227
Name: is_future, dtype: int64

In [7]:
full_df.head()

,key,date,feed_key,base64_url,gtfs_dataset_key,is_future,name,type,regional_feed_type
0,af6d49c345ba965acee24b471b24c952,2023-01-17,c4f6bab7c67286bbf2bf40e51bbd13d1,aHR0cHM6Ly9ndGZzLnNmbXRhLmNvbS90cmFuc2l0ZGF0YS...,025a71429907cc8bd18d33193752234c,False,SFMTA Schedule,schedule,Regional Precursor Feed
1,74a8ea9f9309e17d552b2328ae46b78d,2023-01-17,071827dbd8e30629592144127920fdac,aHR0cHM6Ly90dWxhcmVjb2cub3JnL3RjYWcvZGF0YS1naX...,0329336f4766770c0fb62a4c527a0df9,False,TCRTA Schedule,schedule,None
2,daf5adfce3fa0ffc40111d77d20fd25b,2023-01-17,3e08b3310376486b226aa7ca4e440199,aHR0cHM6Ly9kYXRhLnRyaWxsaXVtdHJhbnNpdC5jb20vZ3...,04f119f76a3a45075c2aa948f68cf29d,False,Vacaville Schedule,schedule,Regional Precursor Feed
3,ff1a7f3ffb5ce010f0733751c4e490b1,2023-01-17,183318cdba4ab20f3e92375ca9eb014b,aHR0cHM6Ly9kYXRhLnRyaWxsaXVtdHJhbnNpdC5jb20vZ3...,05928c3277c0ead4d4aed02325d329f1,False,Desert Roadrunner Schedule,schedule,None
4,f1d2514857508bc2eea1d1823f321846,2023-01-17,54604712e45e9268ba76bfc05766dd0d,aHR0cHM6Ly9mb290aGlsbHRyYW5zaXQucmlkZXJhbGVydH...,07484e2075d31e135baa85b476c851dd,False,Foothill Schedule,schedule,None


In [8]:
def num_rows_and_other_stats(df: pd.DataFrame, feed_option: str):
    """
    Get stats for different filtering to double check.
    """
    subset_df = df >> gtfs_utils_v2.filter_feed_options(feed_option) 
    
    print(f"# rows: {len(subset_df)}")
    print("---------------")
    print(f"regional_feed_type: {subset_df.regional_feed_type.value_counts()}")
    print("---------------")
    print(f"is_future: {subset_df.is_future.value_counts()}")

In [9]:
num_rows_and_other_stats(full_df, "customer_facing")

# rows: 152
---------------
regional_feed_type: Combined Regional Feed    2
Name: regional_feed_type, dtype: int64
---------------
is_future: False    152
Name: is_future, dtype: int64


In [10]:
num_rows_and_other_stats(full_df, "use_subfeeds")

# rows: 186
---------------
regional_feed_type: Regional Subfeed          35
Combined Regional Feed     1
Name: regional_feed_type, dtype: int64
---------------
is_future: False    186
Name: is_future, dtype: int64


In [11]:
num_rows_and_other_stats(full_df, "current_feeds")

# rows: 187
---------------
regional_feed_type: Regional Subfeed          35
Combined Regional Feed     2
Name: regional_feed_type, dtype: int64
---------------
is_future: False    187
Name: is_future, dtype: int64


In [12]:
num_rows_and_other_stats(full_df, "include_precursor") 

# rows: 227
---------------
regional_feed_type: Regional Precursor Feed    40
Regional Subfeed           35
Combined Regional Feed      2
Name: regional_feed_type, dtype: int64
---------------
is_future: False    227
Name: is_future, dtype: int64


In [13]:
num_rows_and_other_stats(full_df, "include_precursor_and_future") 

# rows: 227
---------------
regional_feed_type: Regional Precursor Feed    40
Regional Subfeed           35
Combined Regional Feed      2
Name: regional_feed_type, dtype: int64
---------------
is_future: False    227
Name: is_future, dtype: int64


In [14]:
def display_outputs(df):
    display(df.head())
    print(f"shape: {df.shape}")
    print(f"columns: {df.columns}")
    
    if isinstance(df, gpd.GeoDataFrame):
        print(f"CRS: {df.crs}")

## Step 1: get feeds and orgs for the day

In [15]:
schedule_datasets = gtfs_utils_v2.schedule_daily_feed_to_organization(
    selected_date = analysis_date,
    keep_cols = ["date", "feed_key", "type", 
                 "regional_feed_type", "name"],
    get_df = True,
    feed_option = "use_subfeeds",
) 

display_outputs(schedule_datasets)
print(schedule_datasets.regional_feed_type.value_counts())


,date,feed_key,type,regional_feed_type,name
1,2023-01-17,071827dbd8e30629592144127920fdac,schedule,None,TCRTA Schedule
3,2023-01-17,183318cdba4ab20f3e92375ca9eb014b,schedule,None,Desert Roadrunner Schedule
4,2023-01-17,54604712e45e9268ba76bfc05766dd0d,schedule,None,Foothill Schedule
5,2023-01-17,5ccf86b4334e6c6db3eee03b9f65372c,schedule,Regional Subfeed,Bay Area 511 Sonoma-Marin Area Rail Transit Sc...
6,2023-01-17,8417613331f75be671e07037e7cc2a5d,schedule,None,Yuba-Sutter Schedule


shape: (186, 5)
columns: Index(['date', 'feed_key', 'type', 'regional_feed_type', 'name'], dtype='object')
Regional Subfeed          35
Combined Regional Feed     1
Name: regional_feed_type, dtype: int64


In [16]:
schedule_datasets.regional_feed_type.value_counts()

Regional Subfeed          35
Combined Regional Feed     1
Name: regional_feed_type, dtype: int64

### Set test cases for filtering

In [17]:
test_cases = [
    "Big Blue Bus Schedule", 
    "Metrolink Schedule"
]

test_feed_keys = [
    "008d5112a7e531d0562d26e34d77869d", # Sacramento Schedule
    "f8d3bfd9e780aa3b3ce1340b2116513f" # Long Beach Schedule
]

In [18]:
df_filter_by_name = (
    gtfs_utils_v2.schedule_daily_feed_to_organization(
        selected_date = analysis_date,
        keep_cols = None,
        get_df = False,
        feed_option = "use_subfeeds"
    ) >> gtfs_utils_v2.filter_operator(test_cases, include_name = True)
    >> collect()
)

df_filter_by_name

,key,date,feed_key,base64_url,gtfs_dataset_key,is_future,name,type,regional_feed_type
0,7ab7457165b694acaa4e0e7114fe386c,2023-01-17,90e78003416c5b09f77a9de8f266c2be,aHR0cHM6Ly93d3cubWV0cm9saW5rdHJhaW5zLmNvbS9nbG...,4cd6a8efc6746bccc4ec834d4c4ca841,False,Metrolink Schedule,schedule,None
1,c908315e18a11ca09e5ead11595ee15c,2023-01-17,9d4387dc55091d50c717582348508bae,aHR0cDovL2d0ZnMuYmlnYmx1ZWJ1cy5jb20vY3VycmVudC...,9ad5e9d3c4d5d9da8a246b213cabddd0,False,Big Blue Bus Schedule,schedule,None


In [19]:
df_filter_by_feed_key = (
    gtfs_utils_v2.schedule_daily_feed_to_organization(
        selected_date = analysis_date,
        keep_cols = None,
        get_df = False,
        feed_option = "use_subfeeds"
    ) >> gtfs_utils_v2.filter_operator(test_feed_keys, include_name = False)
    >> collect()
)

df_filter_by_feed_key

,key,date,feed_key,base64_url,gtfs_dataset_key,is_future,name,type,regional_feed_type
0,58d22b05010728b166883376ae9763b2,2023-01-17,008d5112a7e531d0562d26e34d77869d,aHR0cHM6Ly9pcG9ydGFsLnNhY3J0LmNvbS9HVEZTL1NSVE...,ac4cfb9faa274cc4a72f3c96850814a8,False,Sacramento Schedule,schedule,None
1,8e384ef2a07eeb0e0a30884acf7f7409,2023-01-17,f8d3bfd9e780aa3b3ce1340b2116513f,aHR0cHM6Ly9sYnRyYW5zaXQuYm94LmNvbS9zaGFyZWQvc3...,b133a552e7fcbe4319755adf2cd8ad1d,False,Long Beach Schedule,schedule,None


## Step 2: trips

In [53]:
trips = gtfs_utils_v2.get_trips(
    selected_date = analysis_date,
    operator_feeds = test_feed_keys,
    trip_cols = ["feed_key", "trip_id", "trip_key", 
                 "route_id", "route_key", 
                 "shape_array_key", "direction_id",
                 "service_hours", "trip_first_departure_sec", 
                 "trip_last_arrival_sec"
                ],
    get_df = True,
)

In [54]:
display_outputs(trips)

,feed_key,trip_id,trip_key,route_id,route_key,shape_array_key,direction_id,service_hours,trip_first_departure_sec,trip_last_arrival_sec
0,f8d3bfd9e780aa3b3ce1340b2116513f,9646012,533b8078cde956c4253051f0126dce5e,41,7678cda72c78d030af6c68cc5694d9d6,e7aaf911634985435af3aafe6838647d,0,0.98,44880,48420
1,008d5112a7e531d0562d26e34d77869d,1083970,c60214d1cf60194dba57473427ef87fa,138,7c5f941ac70543f8e60b1afa325f29df,3bec19f01935492cf64f388065b699b9,1,0.60,33000,35160
2,f8d3bfd9e780aa3b3ce1340b2116513f,9646156,6f281473e8d2f4020bd719f86374ed8f,71,f9936a3eb2f44faf5521c244a63cc606,c1b9f26a8f118f9f11586839e79cec47,0,1.03,31080,34800
3,008d5112a7e531d0562d26e34d77869d,1083976,cbed93e2ff0af22bbc8c2d3f518d7f36,138,7c5f941ac70543f8e60b1afa325f29df,3bec19f01935492cf64f388065b699b9,1,0.75,54600,57300
4,f8d3bfd9e780aa3b3ce1340b2116513f,9646152,83102a182cb57419c538c11676c46246,71,f9936a3eb2f44faf5521c244a63cc606,c1b9f26a8f118f9f11586839e79cec47,0,0.95,21780,25200


shape: (4245, 10)
columns: Index(['feed_key', 'trip_id', 'trip_key', 'route_id', 'route_key',
       'shape_array_key', 'direction_id', 'service_hours',
       'trip_first_departure_sec', 'trip_last_arrival_sec'],
      dtype='object')


In [22]:
trips2 = gtfs_utils_v2.get_trips(
    selected_date = analysis_date,
    operator_feeds = test_cases,
    trip_cols = ["feed_key", "name", "trip_id",  
                 "route_id", 
                 "shape_id", "shape_array_key", 
                 "direction_id",
                ],
    get_df = True,
)

In [23]:
display_outputs(trips2)

,feed_key,name,trip_id,route_id,shape_id,shape_array_key,direction_id
0,9d4387dc55091d50c717582348508bae,Big Blue Bus Schedule,894627,3563,26193,a794bb868922053ce7ac40c7936c0968,0
1,9d4387dc55091d50c717582348508bae,Big Blue Bus Schedule,894623,3562,26189,74c2812c42a1583a5c1010bab74a233f,1
2,9d4387dc55091d50c717582348508bae,Big Blue Bus Schedule,895162,3565,26205,ca33f7510c1ee7e7846d73268ca233a3,1
3,9d4387dc55091d50c717582348508bae,Big Blue Bus Schedule,894630,3563,26194,fc5c597d51a6692a8cfdd8d85dd63476,1
4,9d4387dc55091d50c717582348508bae,Big Blue Bus Schedule,893222,3554,26161,58164fa5256e4e03e7d3bb1a57a6d0a7,1


shape: (1530, 7)
columns: Index(['feed_key', 'name', 'trip_id', 'route_id', 'shape_id',
       'shape_array_key', 'direction_id'],
      dtype='object')


## Step 2: stops

In [24]:
stops = gtfs_utils_v2.get_stops(
    selected_date = analysis_date,
    operator_feeds = test_feed_keys,
    stop_cols = ["feed_key", "stop_id", "stop_name", 
                 "route_type_3", ],
    get_df = True,
    crs = "EPSG:2229",
)

/opt/conda/lib/python3.9/site-packages/sqlalchemy_bigquery/_types.py:101: SAWarning: Did not recognize type 'GEOGRAPHY' of column 'pt_geom'


In [25]:
display_outputs(stops)

,feed_key,stop_id,stop_name,route_type_3,geometry
0,008d5112a7e531d0562d26e34d77869d,254,GREENHAVEN DR & SUNLIT CIR (SB),11.0,POINT (5551364.398 3482696.311)
1,008d5112a7e531d0562d26e34d77869d,1610,65TH ST & ELDERCREEK RD (SB),36.0,POINT (5578965.012 3481947.860)
2,008d5112a7e531d0562d26e34d77869d,2183,FLORIN RD & GLORIA DR (SB),41.0,POINT (5547251.214 3480775.022)
3,f8d3bfd9e780aa3b3ce1340b2116513f,1936,Wardlow & Delta SE,44.0,POINT (6497603.077 1758246.392)
4,f8d3bfd9e780aa3b3ce1340b2116513f,1062,BELLFLOWER & FLORA VISTA NE,52.0,POINT (6523665.276 1780980.919)


shape: (4718, 5)
columns: Index(['feed_key', 'stop_id', 'stop_name', 'route_type_3', 'geometry'], dtype='object')
CRS: EPSG:2229


## Step 3: shapes

In [48]:
[test_feed_keys[0]]

['008d5112a7e531d0562d26e34d77869d']

In [49]:
shapes = gtfs_utils_v2.get_shapes(
    selected_date = analysis_date,
    operator_feeds = [test_feed_keys[0]],
    shape_cols = ["feed_key", "shape_id", "shape_array_key",
                 "n_trips"],
    get_df = True,
    crs = "EPSG:3310",
)

/opt/conda/lib/python3.9/site-packages/sqlalchemy_bigquery/_types.py:101: SAWarning: Did not recognize type 'GEOGRAPHY' of column 'pt_array'


In [51]:
type(shapes)

NoneType

In [47]:
display_outputs(shapes)

AttributeError: 'NoneType' object has no attribute 'head'

## Step 4: stop_times

In [61]:
# Select from the 2 test cases, the first 5 trip_ids
test_trips = trips2[trips2.name.isin(test_cases)
                  ].trip_id.unique().tolist()[:5]

In [62]:
# Input this as our trip_df
sample_trips = trips2[trips2.trip_id.isin(test_trips)]

In [63]:
# Grab feed key, or else can't subset...
feed_key_for_sample_trips = sample_trips.feed_key.unique().tolist()
feed_key_for_sample_trips

['9d4387dc55091d50c717582348508bae']

In [64]:
stop_times = gtfs_utils_v2.get_stop_times(
    selected_date = analysis_date,
    operator_feeds = feed_key_for_sample_trips,
    stop_time_cols = ["feed_key", "trip_id", "stop_id", 
                      "stop_sequence", 
                      "arrival_sec", "departure_sec"
                ],
    get_df = True,
    trip_df = sample_trips
)

In [65]:
display_outputs(stop_times)

,feed_key,trip_id,stop_id,stop_sequence,arrival_sec,departure_sec,arrival_hour,departure_hour
0,9d4387dc55091d50c717582348508bae,893222,34,2,21704,21704,6,6
1,9d4387dc55091d50c717582348508bae,894630,272,2,65193,65193,18,18
2,9d4387dc55091d50c717582348508bae,894623,886,2,53931,53931,14,14
3,9d4387dc55091d50c717582348508bae,895162,1514,2,55044,55044,15,15
4,9d4387dc55091d50c717582348508bae,894627,5,3,29142,29142,8,8


shape: (156, 8)
columns: Index(['feed_key', 'trip_id', 'stop_id', 'stop_sequence', 'arrival_sec',
       'departure_sec', 'arrival_hour', 'departure_hour'],
      dtype='object')


In [66]:
print(stop_times.arrival_hour.value_counts())
print(stop_times.departure_hour.value_counts())

15    58
8     18
19    17
16    16
6     15
18    15
9     14
14     3
Name: arrival_hour, dtype: int64
15    58
8     18
19    17
16    16
6     15
18    15
9     14
14     3
Name: departure_hour, dtype: int64
